#**NER на основе ячейки GRU**

In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=8bec8288d908f765295ecadc5ddf8776b7a42b85b6b0f37c7a0080d7e697f577
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_from_disk
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report, f1_score

#Инициализация гиперпараметров

In [ ]:
dataset_path = '/content/drive/MyDrive/Colab Notebooks/dl/proj_fashion_ner/fashion_dataset_dict'
save_dir = Path("./gru_ner_ckpt")
save_dir.mkdir(parents=True, exist_ok=True)

seed = 42
batch_size = 64
emb_dim = 128
hidden_size = 256
num_layers = 1
bidirectional = True
dropout = 0.3
lr = 3e-3
max_epochs = 10
grad_clip = 1.0

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
#torch.backends.cudnn.deterministic = True
#torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Загрузка датасета

In [ ]:
ds = load_from_disk(dataset_path)
with open(f"{dataset_path}/label_maps.json", "r", encoding="utf-8") as f:
    maps = json.load(f)

label2id = maps["label2id"]
id2label = {int(k): v for k, v in maps["id2label"].items()}
tag_weight = maps["tag_weight"]
id2label

{0: 'O', 1: 'B-EVENT', 2: 'B-FEAT', 3: 'B-ITEM'}

#Вспомогательные функции

In [ ]:
PAD = "<pad>"
UNK = "<unk>"

def build_vocab(dataset, min_freq=1):
    from collections import Counter
    cnt = Counter()
    for ex in dataset["tokens"]:
        cnt.update(ex)
    itos = [PAD, UNK] + [tok for tok, c in cnt.items() if c >= min_freq]
    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos

stoi, itos = build_vocab(ds["train"])
vocab_size = len(itos)
pad_token_id = stoi[PAD]
unk_token_id = stoi[UNK]
pad_label_id = -100  # для ignore_index в CrossEntropy
print(f"vocabulary size: {vocab_size}")

vocabulary size: 1171


In [ ]:
def numericalize_tokens(tokens):
    return [stoi.get(t, unk_token_id) for t in tokens]

def collate_fn(batch):
    # batch: list of dicts with 'tokens', 'tags'
    lengths = [len(ex["tokens"]) for ex in batch]
    maxlen = max(lengths)
    x = []
    y = []
    mask = []
    for ex in batch:
        ids = numericalize_tokens(ex["tokens"])
        tags = ex["tags"]
        pad_len = maxlen - len(ids)
        x.append(ids + [pad_token_id] * pad_len)
        y.append(tags + [pad_label_id] * pad_len)
        mask.append([1]*len(ids) + [0]*pad_len)
    x = torch.tensor(x, dtype=torch.long)
    y = torch.tensor(y, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    mask = torch.tensor(mask, dtype=torch.bool)
    return {"input_ids": x, "labels": y, "lengths": lengths, "mask": mask}

train_loader = DataLoader(ds["train"], batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(ds["test"], batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

num_labels = len(label2id)

#Модель

In [ ]:
class BiGRUNER(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers, num_labels, pad_idx, bidirectional=True, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            emb_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(out_dim, num_labels)

    def forward(self, input_ids, lengths):
        # input_ids: (B, T), lengths: (B,)
        emb = self.embedding(input_ids)            # (B, T, E)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)  # (B, T, H*)
        out = self.dropout(out)
        logits = self.classifier(out)              # (B, T, num_labels)
        return logits


#Обучение модели без весов классов

In [ ]:
model = BiGRUNER(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_labels=num_labels,
    pad_idx=pad_token_id,
    bidirectional=bidirectional,
    dropout=dropout
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_label_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [ ]:
def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)
        logits = model(x, lengths)
        loss = criterion(logits.view(-1, num_labels), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))

@torch.no_grad()
def evaluate(loader, split_name="valid"):
    model.eval()
    all_preds = []
    all_labels = []
    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)
        mask = batch["mask"].to(device)
        logits = model(x, lengths)
        pred_ids = logits.argmax(-1)

        for pi, yi, mi in zip(pred_ids.cpu().tolist(), y.cpu().tolist(), mask.cpu().tolist()):
            pi = [p for p, m in zip(pi, mi) if m == 1]
            yi = [gt for gt, m in zip(yi, mi) if m == 1]
            # в BIO-теги
            all_preds.append([id2label[int(p)] for p in pi])
            all_labels.append([id2label[int(g)] for g in yi])

    f1 = f1_score(all_labels, all_preds)
    print(f"[{split_name}] seqeval micro-F1: {f1:.4f}")
    print(classification_report(all_labels, all_preds, digits=4))
    return f1

best_f1 = -1.0
best_path = save_dir / "best.pt"

time_start = time.time()

for epoch in range(1, max_epochs + 1):


    tr_loss = train_one_epoch(train_loader)
    print(f"Epoch {epoch}/{max_epochs} | train loss: {tr_loss:.4f}")

    torch.save({"model_state": model.state_dict(),
                    "stoi": stoi,
                    "itos": itos,
                    "config": {
                        "emb_dim": emb_dim,
                        "hidden_size": hidden_size,
                        "num_layers": num_layers,
                        "bidirectional": bidirectional,
                        "dropout": dropout,
                        "num_labels": num_labels,
                        "pad_token_id": pad_token_id
                    }},
                   best_path)

time_stop = time.time()
time_delta = time_stop - time_start
print(f'\ntotal time: {time_delta//60:.0f} min {time_delta % 60:.4f} sec')

Epoch 1/10 | train loss: 0.7561
Epoch 2/10 | train loss: 0.3036
Epoch 3/10 | train loss: 0.1625
Epoch 4/10 | train loss: 0.0945
Epoch 5/10 | train loss: 0.0661
Epoch 6/10 | train loss: 0.0408
Epoch 7/10 | train loss: 0.0265
Epoch 8/10 | train loss: 0.0128
Epoch 9/10 | train loss: 0.0068
Epoch 10/10 | train loss: 0.0038


##Оценка модели

In [ ]:
best_path = save_dir / "best.pt"

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print("Loaded best checkpoint.")
evaluate(test_loader, "test")

Loaded best checkpoint.
[test] seqeval micro-F1: 0.9454
              precision    recall  f1-score   support

       EVENT     0.9792    0.9038    0.9400        52
        FEAT     0.9901    0.8933    0.9393       225
        ITEM     1.0000    0.9172    0.9568       145

   micro avg     0.9922    0.9028    0.9454       422
   macro avg     0.9898    0.9048    0.9454       422
weighted avg     0.9922    0.9028    0.9454       422



np.float64(0.9454094292803971)

##Инференс

In [ ]:
@torch.no_grad()
def predict_tokens(tokens):
    model.eval()
    time_start = time.time()
    ids = torch.tensor([numericalize_tokens(tokens)], dtype=torch.long)
    lengths = torch.tensor([len(tokens)], dtype=torch.long)
    logits = model(ids.to(device), lengths.to(device))
    pred = logits.argmax(-1).squeeze(0).cpu().tolist()[: len(tokens)]
    time_stop = time.time()
    time_delta = time_stop - time_start
    print(f'total time: {time_delta//60:.0f} min {time_delta % 60:.4f} sec')
    return [id2label[int(p)] for p in pred]

print(predict_tokens("очень красивое зеленое платье нравится еще желтые ботинки и красная рубашка".split()))

['O', 'O', 'O', 'B-ITEM', 'O', 'O', 'O', 'B-FEAT', 'O', 'B-FEAT', 'B-ITEM']


In [ ]:
print(predict_tokens('на прошлой неделе купил тулуп и сапоги!))  что можете посоветовать одеть на рыбалку'.split()))

['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ITEM', 'O', 'O', 'O', 'B-EVENT']


#Обучение модели с взвешенными классами

In [ ]:
model = BiGRUNER(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_labels=num_labels,
    pad_idx=pad_token_id,
    bidirectional=bidirectional,
    dropout=dropout
).to(device)

In [ ]:
#расчет весов для классов
from collections import Counter
lab_cnt = Counter()
for tags in ds["train"]["tags"]:
    lab_cnt.update([t for t in tags if t != pad_label_id])
weights = np.zeros(num_labels, dtype=np.float32)
total = sum(lab_cnt.values())
for lid in range(num_labels):
    freq = lab_cnt.get(lid, 1)
    weights[lid] = total / (num_labels * freq)

weights

array([0.33492848, 8.188424  , 1.9788691 , 2.5851479 ], dtype=float32)

In [ ]:
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(ignore_index=pad_label_id, weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [ ]:
best_f1 = -1.0
best_path = save_dir / "weighted_classes_best.pt"

time_start = time.time()

for epoch in range(1, max_epochs + 1):


    tr_loss = train_one_epoch(train_loader)
    print(f"Epoch {epoch}/{max_epochs} | train loss: {tr_loss:.4f}")

    torch.save({"model_state": model.state_dict(),
                    "stoi": stoi,
                    "itos": itos,
                    "config": {
                        "emb_dim": emb_dim,
                        "hidden_size": hidden_size,
                        "num_layers": num_layers,
                        "bidirectional": bidirectional,
                        "dropout": dropout,
                        "num_labels": num_labels,
                        "pad_token_id": pad_token_id
                    }},
                   best_path)

time_stop = time.time()
time_delta = time_stop - time_start
print(f'\ntotal time: {time_delta//60:.0f} min {time_delta % 60:.4f} sec')

Epoch 1/10 | train loss: 0.8562
Epoch 2/10 | train loss: 0.3017
Epoch 3/10 | train loss: 0.1482
Epoch 4/10 | train loss: 0.0835
Epoch 5/10 | train loss: 0.0495
Epoch 6/10 | train loss: 0.0292
Epoch 7/10 | train loss: 0.0216
Epoch 8/10 | train loss: 0.0161
Epoch 9/10 | train loss: 0.0105
Epoch 10/10 | train loss: 0.0080


##Оценка модели

In [ ]:
best_path = save_dir / "weighted_classes_best.pt"

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print("Loaded best checkpoint.")
evaluate(test_loader, "test")

Loaded best checkpoint.
[test] seqeval micro-F1: 0.8947
              precision    recall  f1-score   support

       EVENT     0.6533    0.9423    0.7717        52
        FEAT     0.9765    0.9244    0.9498       225
        ITEM     0.7977    0.9517    0.8679       145

   micro avg     0.8568    0.9360    0.8947       422
   macro avg     0.8092    0.9395    0.8631       422
weighted avg     0.8753    0.9360    0.8997       422



np.float64(0.8946772366930917)

##Инференс

In [ ]:
@torch.no_grad()
def predict_tokens(tokens):
    model.eval()
    ids = torch.tensor([numericalize_tokens(tokens)], dtype=torch.long)
    lengths = torch.tensor([len(tokens)], dtype=torch.long)
    logits = model(ids.to(device), lengths.to(device))
    pred = logits.argmax(-1).squeeze(0).cpu().tolist()[: len(tokens)]
    return [id2label[int(p)] for p in pred]

print(predict_tokens("очень красивое зеленое платье нравится еще желтые ботинки и красная рубашка".split()))

['O', 'O', 'O', 'B-ITEM', 'O', 'O', 'B-FEAT', 'B-ITEM', 'O', 'B-FEAT', 'B-ITEM']


In [ ]:
print(predict_tokens('на прошлой неделе купил тулуп и сапоги!))  что можете посоветовать одеть на рыбалку'.split()))

['O', 'B-EVENT', 'B-ITEM', 'O', 'B-ITEM', 'O', 'O', 'O', 'O', 'O', 'B-ITEM', 'O', 'B-EVENT']
